In [1]:
# Imports
import pandas as pd
import duckdb
import os

# 1. Get Biographies
This notebook writes .txt files of biography main texts.

**Download the following file from from https://zenodo.org/records/20247946 to run this notebook:**
- SpotlightWeightSource_0102_0505_fullmatch_corrected.csv 

## Get sources linking to Napoleon

In [13]:
# Get data with old Spotlight predictions
fullmatch_df = pd.read_csv('../data/SpotlightWeightSource_0102_0505_fullmatch_corrected.csv')
fullmatch_df.head()

,language_code,source_wikidata_id,target_wikidata_id,weight
0,en,Q6882,Q53003,0.0
1,en,Q6882,Q5912,0.0
2,en,Q6882,Q212719,0.0
3,en,Q6882,Q514998,0.0
4,en,Q6882,Q210134,0.0


In [3]:
# Only keep sources with Napoleon (Q517) as target and weight over 0
filtered_df = fullmatch_df[(fullmatch_df["target_wikidata_id"]=="Q517") & (fullmatch_df["weight"]>0.0) & (fullmatch_df["language_code"]=="en")]
filtered_df.shape

(1060, 4)

In [4]:
# Split into low / mid / high quantiles by weight rank
filtered_df['quantile'] = pd.qcut(filtered_df['weight'].rank(method='first'), q=3, labels=['low', 'mid', 'high'])

# Sample evenly - 45 total
groups = {'low': 15, 'mid': 15, 'high': 15}
napo_sample = pd.concat([
    filtered_df[filtered_df['quantile'] == group].sample(n=size, random_state=24)
    for group, size in groups.items()
])

print(napo_sample.groupby('quantile')['weight'].agg(['min', 'max', 'count']))

          min   max  count
quantile                  
low       1.0   1.0     15
mid       1.0   3.0     15
high      3.0  23.0     15


/var/folders/d1/9j4b18194g70dz3d5kkx7vqw0000gn/T/ipykernel_96436/1343385019.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['quantile'] = pd.qcut(filtered_df['weight'].rank(method='first'), q=3, labels=['low', 'mid', 'high'])
/var/folders/d1/9j4b18194g70dz3d5kkx7vqw0000gn/T/ipykernel_96436/1343385019.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(napo_sample.groupby('quantile')['weight'].agg(['min', 'max', 'count']))


## Build graph with sources linking to Napoleon

In [5]:
# Add Napoleon III to the samples
nap3 = filtered_df[filtered_df['source_wikidata_id']=="Q7721"]
napo_sample = pd.concat([napo_sample, nap3])

# Save as dataframe for use in other notebook
napo_sample.to_csv('../data/napo_sample.csv', index=False)

In [8]:
# List of napoleon sources
napo_sources_ids = napo_sample["source_wikidata_id"].values.tolist()

In [ ]:
# Load data with filtered entities
entities = pd.read_csv('../data/entities_filtered_by_languages.csv')

# Only keep people from our sample
napo_sources = entities[entities['wikidata_id'].isin(napo_sources_ids)]

# Save csv file with sample of 45 sources + Napoleon III
napo_sources.to_csv('../data/napo_sources.csv', index=False)
napo_sources.head()

,wikidata_id,birth,bigperiod_birth,un_subregion,gender
519,Q1048,-100.0,1.Ancient History Before 500AD,Southern Europe,Male
2383,Q115995,1795.0,4.Mid Modern Period 1751-1900AD,Western Europe,Male
4437,Q12718,1798.0,4.Mid Modern Period 1751-1900AD,Western Europe,Male
5211,Q130221,1194.0,2.Post-Classical History 501-1500AD,Southern Europe,Male
6346,Q1342645,1739.0,3.Early Modern Period 1501-1750AD,Western Europe,Male


**Run the following command in terminal to build the graph:**

```
python src/main.py build \
    --input data/napo_sources.csv \
    --output data/out/napo_sources.db \
    --languages en
```

## Write text files for manual annotation

In [10]:
con = duckdb.connect('../data/out/napo_sources.db')

rows = con.execute("""
    SELECT wp.language_code, wp.page_title, e.extract
    FROM wiki_page wp
    JOIN extract e ON e.wiki_page_id = wp.wiki_page_id
""").fetchall()

con.close()

output_dirs = [
    "../data/text_files",
    "../hpc_jobs/hpc_relik/data/text_files",
    "../hpc_jobs/hpc_relik_cie/data/text_files",
]

for language_code, page_title, extract in rows:
    filename = f"{page_title.replace(' ', '_')}_{language_code}.txt"
    for d in output_dirs:
        with open(os.path.join(d, filename), "w", encoding="utf-8") as f:
            f.write(extract or "")
    print(f"Wrote {filename}")

Wrote Friedrich_Wilhelm_Freiherr_von_Bülow_en.txt
Wrote Paolo_Virzì_en.txt
Wrote Gaspard_de_Prony_en.txt
Wrote Félicité_de_La_Mennais_en.txt
Wrote Achille_Baraguey_d'Hilliers_en.txt
Wrote Vincent-Marie_Viénot,_Count_of_Vaublanc_en.txt
Wrote Leo_Tolstoy_en.txt
Wrote Franz_Joseph_Gall_en.txt
Wrote Ida_Laura_Pfeiffer_en.txt
Wrote Napoleon_III_en.txt
Wrote Józef_Bem_en.txt
Wrote Archduke_Albrecht,_Duke_of_Teschen_en.txt
Wrote Ferdinand_Foch_en.txt
Wrote Donald_Adamson_en.txt
Wrote François_Christophe_de_Kellermann_en.txt
Wrote Maurice_d'Elbée_en.txt
Wrote Jacques_Laffitte_en.txt
Wrote Antonio_Canova_en.txt
Wrote Oscar_I_of_Sweden_en.txt
Wrote Robert_Clive_en.txt
Wrote Jean_Maximilien_Lamarque_en.txt
Wrote Daniel_O'Connell_en.txt
Wrote Georges_Cadoudal_en.txt
Wrote Mikhail_Tukhachevsky_en.txt
Wrote Prince_Philippe,_Count_of_Paris_en.txt
Wrote Germinal_Pierre_Dandelin_en.txt
Wrote Joseph_Haydn_en.txt
Wrote Frederick_II,_Holy_Roman_Emperor_en.txt
Wrote Prince_Josias_of_Saxe-Coburg-Saalfeld_en

## Get Napoleon File

In [4]:
entities = pd.read_csv('../data/entities_filtered_by_languages.csv')

# Napleon Bonaparte 
napoleon_row = entities[entities['wikidata_id'] == 'Q517']

# Make to csv file
napoleon_row.to_csv('../data/napoleon_self.csv', index=False)

```
python src/main.py build \
    --input data/napoleon_self.csv \
    --output data/out/napoleon_self.db \
    --languages en
```

In [8]:
con = duckdb.connect('../data/out/napoleon_self.db')

rows = con.execute("""
    SELECT wp.language_code, wp.page_title, e.extract
    FROM wiki_page wp 
    JOIN extract e ON e.wiki_page_id = wp.wiki_page_id
    WHERE wp.wikidata_id == 517
""").fetchall()

output_dirs = [
    "../data/text_files",
    "../hpc_jobs/hpc_relik/data/text_napoleon",
    "../hpc_jobs/hpc_relik_cie/data/text_napoleon",
]

for language_code, page_title, extract in rows:
    filename = f"{page_title.replace(' ', '_')}_{language_code}.txt"
    for d in output_dirs:
        with open(os.path.join(d, filename), "w", encoding="utf-8") as f:
            f.write(extract or "")
    print(f"Wrote {filename}")

con.close()

Wrote Napoleon_en.txt


## Check Napoleon I edges

In [14]:
napoleon_edges = fullmatch_df[(fullmatch_df['source_wikidata_id'] == 'Q517') & 
                              (fullmatch_df["language_code"]=="en") & 
                              (fullmatch_df["weight"]>0.0)]
len(napoleon_edges)

67

In [15]:
# Does Anna Pavlovna of Russia have an edge?
napoleon_edges[napoleon_edges['source_wikidata_id'] == 'Q160800']

,language_code,source_wikidata_id,target_wikidata_id,weight
